# BFT65B - Part 5: demo Spark MLlib (Colab)

**SOURCE:** Pipeline MLlib (Imputer, VectorAssembler, StandardScaler, LogisticRegression, GBTClassifier), PR-AUC và việc chỉ fit
trên train theo bài giảng "Big Data, Spark & ML at Scale" [2] (Problem 8-10); chia theo thời gian có khoảng đệm theo [8] (P6).
**AI:** Claude (Anthropic) viết notebook này. Tôi đã đọc, chạy trên Colab và đối chiếu.

**Phạm vi:** chỉ dùng dữ liệu DEVELOPMENT (qi <= 1991Q4), không dùng test. Fold giống fold 1991 của p5_validation.py:
train = các quý có nhãn kết thúc trước 1991Q1 (khoảng đệm 4 quý), validation = 1991Q1-Q4.
MLlib không có bước cắt phân vị (winsorize) nên kết quả chỉ gần, không trùng, với scikit-learn.

**Cách chạy:** Runtime > Run all. Ở cell 2, chọn 2 file: `p5_spark_dev.parquet` (trong `data/`) và `p5_dev_selection.json` (trong `reports/`).
Cuối cùng tải về `p5_spark_results.json` (đặt vào `reports/` của repo).

In [1]:
# Cell 1: Colab đã có Java 21; PySpark 4.x chạy được với Java 17/21.
import os
os.environ.pop("JAVA_HOME", None)
!java -version 2>&1 | grep -i "openjdk version"
!pip install -q pyspark
!pip show pyspark | grep -i "^version"

openjdk version "21.0.12.1" 2026-08-18
Version: 4.0.5


In [2]:
# Cell 2: tải 2 file từ máy lên Colab
from google.colab import files
up = files.upload()
print({k: round(len(v) / 1e6, 1) for k, v in up.items()}, "MB")

Saving p5_dev_selection.json to p5_dev_selection.json
Saving p5_spark_dev.parquet to p5_spark_dev.parquet
{'p5_dev_selection.json': 0.0, 'p5_spark_dev.parquet': 102.3} MB


In [3]:
# Cell 3: khởi động Spark, đọc dữ liệu, chia train/validation theo thời gian có khoảng đệm 4 quý
import json, time, os
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]").appName("bft65b_p5")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "8").getOrCreate())
print("Spark", spark.version, "| CPU máy ảo:", os.cpu_count())

df = spark.read.parquet("p5_spark_dev.parquet").withColumn("y", F.col("y").cast("double"))
FEATS = [c for c in df.columns if c not in ("cert", "qi", "y")]
V0 = 1991 * 4
train = df.filter(F.col("qi") <= V0 - 5)                       # nhãn của dòng train kết thúc trước khi validation bắt đầu
valid = df.filter((F.col("qi") >= V0) & (F.col("qi") <= V0 + 3))
print("đặc trưng:", len(FEATS))
for name, d in (("train", train), ("validation 1991", valid)):
    r = d.agg(F.count(F.lit(1)).alias("n"), F.sum("y").alias("pos")).first()
    print(f"{name:<16} {r['n']:,} dòng, {int(r['pos']):,} dòng y=1")

Spark 4.0.5 | CPU máy ảo: 2
đặc trưng: 29
train            418,728 dòng, 6,075 dòng y=1
validation 1991  59,925 dòng, 777 dòng y=1


In [4]:
# Cell 4: hai Pipeline MLlib (Imputer -> Assembler -> Scaler -> mô hình); fit CHỈ trên train
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def build(model):
    imp = Imputer(inputCols=FEATS, outputCols=[c + "_i" for c in FEATS], strategy="median")
    asm = VectorAssembler(inputCols=[c + "_i" for c in FEATS], outputCol="raw")
    scl = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True)
    return Pipeline(stages=[imp, asm, scl, model])

models = {
    "lr": LogisticRegression(featuresCol="features", labelCol="y", maxIter=100, regParam=1e-4),
    "gbt": GBTClassifier(featuresCol="features", labelCol="y", maxIter=50, maxDepth=4, seed=42),
}
ev_pr = BinaryClassificationEvaluator(labelCol="y", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
ev_roc = BinaryClassificationEvaluator(labelCol="y", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mllib = {}
for name, m in models.items():
    t = time.perf_counter()
    fitted = build(m).fit(train)                               # mọi tham số học (trung vị, trung bình, độ lệch chuẩn, trọng số) từ train
    pred = fitted.transform(valid)
    mllib[name] = {"pr_auc": ev_pr.evaluate(pred), "roc_auc": ev_roc.evaluate(pred), "fit_and_score_seconds": round(time.perf_counter() - t, 1)}
    print(name, mllib[name])

lr {'pr_auc': 0.20713023554858512, 'roc_auc': 0.9399606545072129, 'fit_and_score_seconds': 49.5}
gbt {'pr_auc': 0.30879741683935913, 'roc_auc': 0.9710458001693547, 'fit_and_score_seconds': 71.8}


In [5]:
# Cell 5: so với scikit-learn (cùng fold 1991, đã chạy ở p5_validation.py)
sel = json.load(open("p5_dev_selection.json"))
skl = {t: next(f for f in sel["per_fold"][t] if f["block"] == "1991-1991") for t in ("lr", "hgb")}
print(f"{'mô hình':<28}{'PR-AUC':>9}{'ROC-AUC':>9}")
print(f"{'scikit-learn LR (đã chọn)':<28}{skl['lr']['pr_auc']:>9.3f}{skl['lr']['roc_auc']:>9.3f}")
print(f"{'scikit-learn HGB (đã chọn)':<28}{skl['hgb']['pr_auc']:>9.3f}{skl['hgb']['roc_auc']:>9.3f}")
print(f"{'MLlib LogisticRegression':<28}{mllib['lr']['pr_auc']:>9.3f}{mllib['lr']['roc_auc']:>9.3f}")
print(f"{'MLlib GBTClassifier':<28}{mllib['gbt']['pr_auc']:>9.3f}{mllib['gbt']['roc_auc']:>9.3f}")

mô hình                        PR-AUC  ROC-AUC
scikit-learn LR (đã chọn)       0.288    0.967
scikit-learn HGB (đã chọn)      0.402    0.971
MLlib LogisticRegression        0.207    0.940
MLlib GBTClassifier             0.309    0.971


In [6]:
# Cell 6: lưu và tải về kết quả (đặt vào reports/ của repo)
out = {"spark": spark.version, "cpu_cores_vm": os.cpu_count(), "fold": "1991-1991 (development)",
       "train_rows": train.count(), "valid_rows": valid.count(), "mllib": mllib,
       "sklearn_same_fold": {"lr": skl["lr"], "hgb": skl["hgb"]}}
json.dump(out, open("p5_spark_results.json", "w"), indent=1)
from google.colab import files
files.download("p5_spark_results.json")
print(out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'spark': '4.0.5', 'cpu_cores_vm': 2, 'fold': '1991-1991 (development)', 'train_rows': 418728, 'valid_rows': 59925, 'mllib': {'lr': {'pr_auc': 0.20713023554858512, 'roc_auc': 0.9399606545072129, 'fit_and_score_seconds': 49.5}, 'gbt': {'pr_auc': 0.30879741683935913, 'roc_auc': 0.9710458001693547, 'fit_and_score_seconds': 71.8}}, 'sklearn_same_fold': {'lr': {'block': '1991-1991', 'n_train': 418728, 'pos_train': 6075, 'n': 59925, 'pos': 777, 'pr_auc': 0.28831511487777084, 'roc_auc': 0.9673998622568313}, 'hgb': {'block': '1991-1991', 'n_train': 418728, 'pos_train': 6075, 'n': 59925, 'pos': 777, 'pr_auc': 0.4023132428518249, 'roc_auc': 0.9714514531921714}}}
